
# TF‑IDF → LibSVM → SageMaker XGBoost (5‑class) — Large & Imbalanced, Multi‑CSV in S3

**Updated:** 2025-10-13 03:34 UTC  

End‑to‑end pipeline for **any 5‑label** text classification dataset stored as **multiple CSV files in S3**:
1. **Processing job** (SKLearnProcessor): read `text` + `label` columns from many CSVs, **TF‑IDF**, split train/val/test, **compute class weights**, and write **LibSVM with instance weights** (`label:weight idx:val …`) to S3.  
2. **Training job** (built‑in **SageMaker XGBoost**): objective=`multi:softprob`, `num_class=5`, early stopping on validation channel.  
3. (Optional) **Batch transform** or deploy endpoint.

**Why LibSVM?** It’s compact for sparse TF‑IDF and natively supports per‑row **instance weights** for SageMaker XGBoost.


In [ ]:

!pip -q install sagemaker==2.* boto3 pandas scikit-learn scipy


In [ ]:

import os, json, boto3, sagemaker, numpy as np, pandas as pd
from sagemaker import Session
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.inputs import TrainingInput
from sagemaker.xgboost.estimator import XGBoost
from sagemaker import get_execution_role

sess = sagemaker.Session()
region = boto3.Session().region_name
try:
    role = get_execution_role()
except Exception:
    role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN", "arn:aws:iam::<account-id>:role/<your-sagemaker-execution-role>")

print("Region:", region)
print("Role:", role)
print("Default bucket:", sess.default_bucket())


In [ ]:

# ==== USER CONFIG ============================================================
BUCKET = sess.default_bucket()
PROJECT = "tfidf-xgb-text-any"
S3_PREFIX_RAW = f"{PROJECT}/raw"
S3_PREFIX_PROC = f"{PROJECT}/processed"
S3_PREFIX_OUT = f"{PROJECT}/output"

S3_URI_RAW = f"s3://{BUCKET}/{S3_PREFIX_RAW}"

TEXT_COL = "text"
LABEL_COL = "label"

NUM_CLASSES = 5

VAL_SIZE = 0.10
TEST_SIZE = 0.10
RANDOM_STATE = 42

TFIDF_KW = dict(
    stop_words="english",
    ngram_range=(1,2),
    max_features=100_000,
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    smooth_idf=True,
    dtype="float32",
)

PROC_INSTANCE_TYPE = "ml.m5.xlarge"
PROC_INSTANCE_COUNT = 1

XGB_FRAMEWORK_VERSION = "1.7-1"
TRAIN_INSTANCE_TYPE = "ml.m5.4xlarge"
TRAIN_INSTANCE_COUNT = 1

USE_CLASS_WEIGHTS = True
SAMPLE_SIZE = None

print("RAW prefix:", S3_URI_RAW)


In [ ]:

PROC_SCRIPT_PATH = "preprocess_tfidf_to_libsvm.py"
with open(PROC_SCRIPT_PATH, "w", encoding="utf-8") as f:
    f.write(proc_script)
print("Wrote", PROC_SCRIPT_PATH)


In [ ]:

sk_proc = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=PROC_INSTANCE_TYPE,
    instance_count=PROC_INSTANCE_COUNT,
    base_job_name=f"{PROJECT}-prep",
    sagemaker_session=sess,
)

proc_inputs = [ProcessingInput(
    source=S3_URI_RAW,
    destination="/opt/ml/processing/input",
    s3_data_distribution_type="ShardedByS3Key",
)]
proc_outputs = [ProcessingOutput(
    output_name="processed",
    source="/opt/ml/processing/output",
    destination=f"s3://{BUCKET}/{S3_PREFIX_PROC}",
)]

sk_proc.run(
    code=PROC_SCRIPT_PATH,
    inputs=proc_inputs,
    outputs=proc_outputs,
    arguments=[
        "--text-col", TEXT_COL,
        "--label-col", LABEL_COL,
        "--num-classes", str(NUM_CLASSES),
        "--val-size", str(VAL_SIZE),
        "--test-size", str(TEST_SIZE),
        "--random-state", str(RANDOM_STATE),
        "--use-class-weights" if USE_CLASS_WEIGHTS else "",
        "--tfidf-kw", json.dumps(TFIDF_KW),
    ],
    wait=False
)
print("Launched processing job. Output →", f"s3://{BUCKET}/{S3_PREFIX_PROC}")


In [ ]:

S3_TRAIN = f"s3://{BUCKET}/{S3_PREFIX_PROC}/train.libsvm"
S3_VALID = f"s3://{BUCKET}/{S3_PREFIX_PROC}/validation.libsvm"

xgb = XGBoost(
    framework_version=XGB_FRAMEWORK_VERSION,
    role=role,
    instance_type=TRAIN_INSTANCE_TYPE,
    instance_count=TRAIN_INSTANCE_COUNT,
    output_path=f"s3://{BUCKET}/{S3_PREFIX_OUT}",
    base_job_name=f"{PROJECT}-xgb",
    sagemaker_session=sess,
    hyperparameters={
        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "num_round": 3000,
        "early_stopping_rounds": 75,
        "eta": 0.03,
        "max_depth": 8,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
    },
)
xgb.fit({"train": TrainingInput(S3_TRAIN, content_type="libsvm"),
         "validation": TrainingInput(S3_VALID, content_type="libsvm")},
        wait=False)
print("Launched training job. Output →", f"s3://{BUCKET}/{S3_PREFIX_OUT}")



### Notes
- For CSV with per-row weights instead of LibSVM, enable hyperparameter **`csv_weights=1`** and put `label,weight,feature1,...` in each row.  
- SageMaker XGBoost hyperparameters (including `num_class`, `num_round`, `csv_weights`, `early_stopping_rounds`) are documented in AWS.  
- Point **training channels** at an S3 **prefix** containing many small files to speed up sharded downloads.
